# Step 7: Test Deployment

Test the complete lakehouse agent system end-to-end.

## Prerequisites

- ✅ Run `06-deploy-agent.ipynb` first
- ✅ All components deployed

## What This Notebook Does

1. Tests OAuth token generation
2. Tests agent invocation
3. Validates end-to-end flow
4. Verifies Lake Formation RLS

## Next Steps

- Run Streamlit UI: `streamlit run streamlit-ui/streamlit_app.py`

In [ ]:
import boto3
import json
import base64
import requests
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
from config import config

ssm_client = boto3.client('ssm')
agentcore_client = boto3.client('bedrock-agentcore')

print('✅ Setup complete')

## Step 1: Get OAuth Token from Cognito

In [ ]:
# Get Cognito configuration
COGNITO_DOMAIN = ssm_client.get_parameter(Name='lh_cognito_domain')['Parameter']['Value']
CLIENT_ID = ssm_client.get_parameter(Name='lh_cognito_app_client_id')['Parameter']['Value']
CLIENT_SECRET = ssm_client.get_parameter(Name='lh_cognito_app_client_secret', WithDecryption=True)['Parameter']['Value']

# Request token
token_url = f'{COGNITO_DOMAIN}/oauth2/token'
credentials = f'{CLIENT_ID}:{CLIENT_SECRET}'
encoded_credentials = base64.b64encode(credentials.encode()).decode()

headers = {
    'Authorization': f'Basic {encoded_credentials}',
    'Content-Type': 'application/x-www-form-urlencoded'
}

data = {
    'grant_type': 'client_credentials',
    'scope': 'lakehouse-api/claims.query'
}

response = requests.post(token_url, headers=headers, data=data)

if response.status_code == 200:
    token_data = response.json()
    ACCESS_TOKEN = token_data['access_token']
    print('✅ OAuth token obtained successfully!')
    print(f'   Token type: {token_data.get("token_type")}')
    print(f'   Expires in: {token_data.get("expires_in")} seconds')
else:
    print(f'❌ Failed to get token: {response.status_code}')
    print(response.text)
    ACCESS_TOKEN = None

## Step 2: Test Agent Invocation

In [ ]:
if ACCESS_TOKEN:
    # Get agent runtime ARN
    AGENT_RUNTIME_ARN = ssm_client.get_parameter(Name='lh_lakehouse_agent_runtime_arn')['Parameter']['Value']
    
    # Prepare payload
    payload = {
        'prompt': 'Show me all my claims',
        'bearer_token': ACCESS_TOKEN
    }
    
    print(f'🤖 Invoking agent...')
    print(f'   Runtime ARN: {AGENT_RUNTIME_ARN}')
    print(f'   Prompt: {payload["prompt"]}')
    
    try:
        response = agentcore_client.invoke_agent_runtime(
            agentRuntimeArn=AGENT_RUNTIME_ARN,
            runtimeSessionId='test-session-001',
            payload=json.dumps(payload).encode('utf-8')
        )
        
        # Parse response
        if 'payload' in response:
            response_payload = json.loads(response['payload'].read().decode('utf-8'))
            print('\n✅ Agent response:')
            print(json.dumps(response_payload, indent=2))
        else:
            print('\n⚠️  No payload in response')
            print(response)
    except Exception as e:
        print(f'\n❌ Error invoking agent: {e}')
else:
    print('⚠️  Skipping agent test - no access token')

## Step 3: Verify Lake Formation RLS

Test that users can only see their own data.

In [ ]:
# This would require testing with different user tokens
print('📋 Lake Formation RLS Verification:')
print('   To fully test RLS:')
print('   1. Get tokens for different users (user001, user002)')
print('   2. Query claims with each token')
print('   3. Verify each user sees only their own claims')
print('\n   Use the Streamlit UI for interactive testing!')

## Step 4: Check CloudWatch Logs

In [ ]:
print('📊 CloudWatch Logs to Check:')
print('\n1. Interceptor Lambda:')
print('   aws logs tail /aws/lambda/lakehouse-gateway-interceptor --follow')
print('\n2. MCP Server:')
print('   Check AgentCore Runtime logs in CloudWatch')
print('\n3. Look for:')
print('   ✅ Bearer token extracted from MCP gateway request')
print('   ✅ Extracted user principal: user@example.com')
print('   ✅ Request authorized for user: user@example.com')

## Summary

✅ **Testing Complete!**

**What was tested:**
- OAuth token generation from Cognito
- Agent invocation with bearer token
- End-to-end request flow

**Next Steps:**

1. **Run Streamlit UI for interactive testing:**
   ```bash
   cd streamlit-ui
   streamlit run streamlit_app.py
   ```

2. **Test with different users:**
   - user001@example.com / TempPass123!
   - user002@example.com / TempPass123!
   - adjuster001@example.com / TempPass123!

3. **Verify Row-Level Security:**
   - Each user should see only their own claims
   - user001 sees 4 claims
   - user002 sees 5 claims

**Troubleshooting:**
- Check CloudWatch logs for errors
- Verify SSM parameters are set correctly
- Ensure all components are deployed